# 1장 2강: 좋은 지표의 조건 — 실습문제

## 실습 목표

- Ravenstack의 비즈니스 목표에 맞춰 KPI, 보조 지표, 허무 지표를 구분할 수 있다.
- 구독 데이터에서 현재 반복 매출과 이탈률을 계산할 수 있다.
- 요금제 또는 유입 경로별 지표를 비교하여 개선이 필요한 대상을 찾을 수 있다.
- 행동 가능성, 비교 가능성, 이해 용이성을 기준으로 지표의 적절성을 판단할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

이번 실습에서는 Ravenstack의 핵심 목표를 다음과 같이 가정합니다.

> **유료 구독을 안정적으로 유지하면서 반복 매출을 늘린다.**

주요 컬럼은 다음과 같습니다.

| 테이블 | 컬럼 | 의미 |
|---|---|---|
| accounts | `account_id` | 고객사 식별자 |
| accounts | `referral_source` | 고객사가 유입된 경로 |
| accounts | `signup_date` | 고객사 가입일 |
| accounts | `churn_flag` | 고객사 이탈 여부 |
| subscriptions | `subscription_id` | 구독 식별자 |
| subscriptions | `plan_tier` | 구독 요금제 |
| subscriptions | `mrr_amount` | 월간 반복 매출(MRR) |
| subscriptions | `is_trial` | 체험 구독 여부 |
| subscriptions | `end_date` | 구독 종료일 |
| subscriptions | `churn_flag` | 구독 이탈 여부 |

> 비율은 별도 지시가 없으면 소수점 둘째 자리의 백분율로 출력합니다.

In [2]:
# 프로덕트 분석 : 사용자 행동 데이터를 분석해 서비스의 문제를 찾고 개선을 위한 의사결정을 돕는 과정
# -> 쇼핑몰 방문자는 늘었는데 구매자는 비슷. 상품조회(클릭 수), 장바구니, 결제 순서등의 사용자 행동을 살펴보면
# 어느 과정에서 딜레이가 있었는지 어느 부분을 개선해야할지 정할 수 있음

# KPI : 비즈니스의 핵심 목표가 얼마나 달성되고 있는지 판단하기 위해 중요하게 관리하는 지표
# -> 월간 유로 강의 구매자수가 KPI가 될 수 있음

# 보조 지표 : KPI의 변화와 그 배경을 구체적으로 이해하기 위해 함께 확인하는 지표
# 허무 지표 : 숫자나 내용 자체는 좋아보이지만 현재 목표나 KPI와는 무관하거나 연결이 약한 지표

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 두 CSV 파일을 각각 `accounts`, `subscriptions`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`를 날짜형으로 변환하세요.
6. `mrr_amount`의 기초 통계량과 `plan_tier`의 빈도를 확인하세요.

In [3]:
# 실습 준비 코드를 작성하세요.

import pandas as pd

# 1. 라이브러리 불러오기 (이미 위에서 pandas as pd)

# 2. CSV 불러오기
accounts = pd.read_csv(r"C:\first-project\product\ravenstack_accounts.csv",encoding="utf-8-sig")
subscriptions = pd.read_csv("ravenstack_subscriptions.csv")

# 3. 행/열 개수, 상위 5행 확인
print("accounts shape:", accounts.shape)
print(accounts.head())
print("\nsubscriptions shape:", subscriptions.shape)
print(subscriptions.head())

# 4. 자료형, 결측치 개수 확인
print("\naccounts dtypes:\n", accounts.dtypes)
print("accounts 결측치:\n", accounts.isna().sum())
print("\nsubscriptions dtypes:\n", subscriptions.dtypes)
print("subscriptions 결측치:\n", subscriptions.isna().sum())

# 5. 날짜형 변환
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"])
subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"])

# 6. mrr_amount 기초통계, plan_tier 빈도
print("\nmrr_amount 기초통계:\n", subscriptions["mrr_amount"].describe())
print("\nplan_tier 빈도:\n", subscriptions["plan_tier"].value_counts())

accounts shape: (500, 10)
  account_id account_name    industry country signup_date referral_source  \
0   A-2e4581    Company_0      EdTech      US  2024-10-16         partner   
1   A-43a9e3    Company_1     FinTech      IN  2023-08-17           other   
2   A-0a282f    Company_2    DevTools      US  2024-08-27         organic   
3   A-1f0ac7    Company_3  HealthTech      UK  2023-08-27           other   
4   A-ce550d    Company_4  HealthTech      US  2024-10-27           event   

    plan_tier  seats  is_trial  churn_flag  
0       Basic      9     False       False  
1       Basic     18     False        True  
2       Basic      1     False       False  
3       Basic     24      True       False  
4  Enterprise     35     False        True  

subscriptions shape: (5000, 14)
  subscription_id account_id  start_date    end_date   plan_tier  seats  \
0        S-8cec59   A-3c1a3f  2023-12-23  2024-04-12  Enterprise     14   
1        S-0f6f44   A-9b9fe9  2024-06-11         NaN      

---

## 필수 1. 비즈니스 목표에 맞는 지표 구분하기

### 문제 1-1. 현재 반복 매출을 중심으로 지표를 계산하고 분류하기

#### 문제 설명

Ravenstack은 **유료 구독을 안정적으로 유지하면서 반복 매출을 늘리는 것**을 핵심 목표로 정했습니다.

다음 세 지표를 직접 계산한 뒤 현재 목표를 기준으로 KPI, 보조 지표, 허무 지표로 분류하세요.

- 현재 유료 구독 MRR
- 전체 구독 이탈률
- 누적 가입 고객사 수

이번 실습에서는 `end_date`가 비어 있고 `is_trial`이 `False`인 구독을 **현재 활성 유료 구독**으로 정의합니다.

#### 요구사항

1. 현재 활성 유료 구독 조건을 불리언 변수 `active_paid_mask`로 만드세요.
2. 현재 활성 유료 구독의 `mrr_amount` 합계를 `current_paid_mrr`로 계산하세요.
3. `subscriptions`의 `churn_flag` 평균으로 `subscription_churn_rate`를 계산하세요.
4. `accounts`의 고유한 `account_id` 개수를 `cumulative_accounts`로 계산하세요.
5. 세 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 현재 비즈니스 목표를 기준으로 다음과 같이 분류하고 이유를 설명하세요.
   - 현재 유료 구독 MRR: KPI
   - 전체 구독 이탈률: 보조 지표
   - 누적 가입 고객사 수: 허무 지표 후보

#### 해석 질문

**Q1.** 현재 유료 구독 MRR을 KPI로 볼 수 있는 이유는 무엇인가요?  
**Q2.** 전체 구독 이탈률은 현재 유료 구독 MRR을 이해하는 데 어떻게 도움을 주나요?  
**Q3.** 누적 가입 고객사 수만으로 현재 서비스가 성장한다고 판단하기 어려운 이유는 무엇인가요?

#### 제출 결과

- 세 지표의 계산 코드와 결과
- KPI, 보조 지표, 허무 지표 분류
- 분류 이유
- Q1~Q3 답변

In [4]:
# 필수 1 코드를 작성하세요.

# 1. 현재 활성 유료 구독 조건
active_paid_mask = subscriptions["end_date"].isna() & (subscriptions["is_trial"] == False)

# 2. 현재 유료 구독 MRR
current_paid_mrr = subscriptions.loc[active_paid_mask, "mrr_amount"].sum()

# 3. 전체 구독 이탈률
subscription_churn_rate = subscriptions["churn_flag"].mean()

# 4. 누적 가입 고객사 수
cumulative_accounts = accounts["account_id"].nunique()

# 5. 출력
print(f"현재 유료 구독 MRR: ${current_paid_mrr:,.2f}")
print(f"전체 구독 이탈률: {subscription_churn_rate * 100:.2f}%")
print(f"누적 가입 고객사 수: {cumulative_accounts:,}개")

# 6. KPI / 보조 지표 / 허무 지표 분류
metric_classification = {
    "현재 유료 구독 MRR": {
        "값": f"${current_paid_mrr:,.2f}",
        "분류": "KPI"
    },
    "전체 구독 이탈률": {
        "값": f"{subscription_churn_rate * 100:.2f}%",
        "분류": "보조 지표",
    },
    "누적 가입 고객사 수": {
        "값": f"{cumulative_accounts:,}개",
        "분류": "허무 지표 후보",
    },
}

print("\n" + "=" * 60)
print("지표 분류")
print("=" * 60)
for metric_name, info in metric_classification.items():
    print(f"\n[{metric_name}] {info['값']}")
    print(f"  분류: {info['분류']}")

현재 유료 구독 MRR: $10,159,608.00
전체 구독 이탈률: 9.72%
누적 가입 고객사 수: 500개

지표 분류

[현재 유료 구독 MRR] $10,159,608.00
  분류: KPI

[전체 구독 이탈률] 9.72%
  분류: 보조 지표

[누적 가입 고객사 수] 500개
  분류: 허무 지표 후보


### 필수 1 답변 작성란

**Q1.** 현재 유료 구독 MRR을 KPI로 볼 수 있는 이유는 무엇인가요?  
-> 현재 활성 유료 구독이 매달 만들어내는 반복 매출을 직접 나타내므로ravenstack의 핵심 목표와 일치

**Q2.** 전체 구독 이탈률은 현재 유료 구독 MRR을 이해하는 데 어떻게 도움을 주나요?  
-> 이탈율이 높아지면 유료 구독과 반복 매출이 감소할 가능성이 있음

**Q3.** 누적 가입 고객사 수만으로 현재 서비스가 성장한다고 판단하기 어려운 이유는 무엇인가요?  
-> 누적 가입 고객사 수는 이미 이탈했거나, 구독을 삭제로 진행하지 않는 고객사들도 포함될 수 있음



---

## 필수 2. 요금제별 핵심 지표 비교하기

### 문제 2-1. 어느 요금제를 우선적으로 살펴봐야 하는가?

#### 문제 설명

전체 평균만 확인하면 요금제별 차이를 놓칠 수 있습니다.  
`Basic`, `Pro`, `Enterprise` 요금제별로 구독 규모, 이탈률, 현재 유료 구독 MRR을 비교하세요.

#### 요구사항

1. `subscriptions`에 `is_active_paid` 컬럼을 만드세요.
   - `end_date`가 비어 있고 `is_trial`이 `False`이면 `True`
2. `plan_tier`별로 다음 값을 집계하여 `plan_metrics`를 만드세요.
   - 전체 구독 수
   - 이탈 구독 수
   - 구독 이탈률
   - 현재 활성 유료 구독 수
3. 현재 활성 유료 구독만 사용해 요금제별 MRR 합계를 계산하고 `active_mrr` 컬럼으로 추가하세요.
4. 이탈률은 백분율로, MRR은 천 단위 구분 기호를 사용하여 출력하세요.
5. 현재 유료 구독 MRR이 가장 큰 요금제와 이탈률이 가장 높은 요금제를 확인하세요.
6. 현재 목표를 고려하여 우선적으로 점검할 요금제 하나를 정하고, 데이터 근거와 확인할 개선 방향을 설명하세요.

#### 해석 질문

**Q1.** 요금제별 이탈률을 비교하는 것이 전체 이탈률만 확인하는 것보다 행동 가능성이 높은 이유는 무엇인가요?  
**Q2.** 현재 유료 구독 MRR이 가장 큰 요금제는 무엇인가요?  
**Q3.** 이탈률이 가장 높은 요금제는 무엇인가요?  
**Q4.** 위 두 결과를 함께 보면 어떤 요금제를 우선적으로 점검할 수 있으며, 그 이유는 무엇인가요?

#### 제출 결과

- `plan_metrics` 집계 코드와 결과
- MRR 및 이탈률 기준 요금제 비교
- 우선 점검 대상과 개선 방향
- Q1~Q4 답변

In [5]:
# 필수 2 코드를 작성하세요.
# 1. is_active_paid 컬럼
subscriptions["is_active_paid"] = subscriptions["end_date"].isna() & (subscriptions["is_trial"] == False)

# 2. plan_tier별 집계
plan_metrics = subscriptions.groupby("plan_tier").agg(
    total_subs=("plan_tier", "count"),
    churned_subs=("churn_flag", "sum"),
    active_paid_subs=("is_active_paid", "sum"),
)
plan_metrics["churn_rate"] = plan_metrics["churned_subs"] / plan_metrics["total_subs"]

# 3. 요금제별 active_mrr
active_mrr = subscriptions.loc[subscriptions["is_active_paid"], :].groupby("plan_tier")["mrr_amount"].sum()
plan_metrics["active_mrr"] = active_mrr

plan_metrics = plan_metrics[["total_subs", "churned_subs", "churn_rate", "active_paid_subs", "active_mrr"]]

# 4. 보기 좋은 형식으로 출력
display_df = plan_metrics.copy()
display_df["churn_rate"] = (display_df["churn_rate"] * 100).round(2).astype(str) + "%"
display_df["active_mrr"] = display_df["active_mrr"].apply(lambda x: f"${x:,.2f}")
print(display_df)

# 5. 확인
top_mrr_plan = plan_metrics["active_mrr"].idxmax()
top_churn_plan = plan_metrics["churn_rate"].idxmax()
print(f"\n현재 유료 구독 MRR이 가장 큰 요금제: {top_mrr_plan} (${plan_metrics.loc[top_mrr_plan, 'active_mrr']:,.2f})")
print(f"이탈률이 가장 높은 요금제: {top_churn_plan} ({plan_metrics.loc[top_churn_plan, 'churn_rate']*100:.2f}%)")

            total_subs  churned_subs churn_rate  active_paid_subs  \
plan_tier                                                           
Basic             1602           152      9.49%              1228   
Enterprise        1723           172      9.98%              1304   
Pro               1675           162      9.67%              1282   

               active_mrr  
plan_tier                  
Basic         $687,914.00  
Enterprise  $7,546,876.00  
Pro         $1,924,818.00  

현재 유료 구독 MRR이 가장 큰 요금제: Enterprise ($7,546,876.00)
이탈률이 가장 높은 요금제: Enterprise (9.98%)


### 필수 2 답변 작성란

**Q1.** 요금제별 이탈률을 비교하는 것이 전체 이탈률만 확인하는 것보다 행동 가능성이 높은 이유는 무엇인가요?  
-> 전체 이탈률은 서비스 전체의 현상을 보여주고, 요금제별 이탈률은 어느 요금제를 먼저 점검해야하는지를 판단  
-> 다음 회의가 열린다면 담당 팀이 개선 대상을 명확하게 설정하여 개선점을 진행할 수 있음

**Q2.** 현재 유료 구독 MRR이 가장 큰 요금제는 무엇인가요?  
-> Enterprise / 유료 구독료 : $7,546,876.00

**Q3.** 이탈률이 가장 높은 요금제는 무엇인가요?  
-> Enterprise / 이탈률 : 9,98%

**Q4.** 위 두 결과를 함께 보면 어떤 요금제를 우선적으로 점검할 수 있으며, 그 이유는 무엇인가요?  
-> Enterprise를 우선적으로 점검  
-> 현재 MRR기여도가 가장 크면서 이탈률이 가장 높아 이 그룹의 이탈률을 줄이는 것이 다음달 매출의 방어 효과가 가장 큼


---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 유입 경로별 고객사 이탈률 비교하기

#### 문제 설명

Ravenstack 마케팅팀은 고객사가 유입된 경로에 따라 이탈 수준이 다른지 확인하려고 합니다. 유입 경로별 고객사 수와 이탈률을 비교하여 우선 점검할 유입 경로를 찾으세요.

> 이 과제는 필수 2에서 수행한 그룹별 지표 집계와 해석을 새로운 컬럼에 동일하게 적용하는 문제입니다.

#### 요구사항

1. `accounts`를 `referral_source`별로 그룹화하세요.
2. 다음 값을 집계하여 `source_metrics`를 만드세요.
   - 전체 고객사 수
   - 이탈 고객사 수
   - 고객사 이탈률
3. 고객사 이탈률이 높은 순서로 정렬하세요.
4. 이탈률은 소수점 둘째 자리의 백분율로 출력하세요.
5. 이탈률이 가장 높은 유입 경로를 찾으세요.
6. 해당 유입 경로를 우선 점검 대상으로 정하고, 확인할 개선 방향을 한 가지 제안하세요.
7. 유입 경로별 이탈률을 좋은 지표의 세 조건으로 평가하세요.
   - 행동 가능성
   - 비교 가능성
   - 이해 용이성

#### 해석 질문

**Q1.** 고객사 이탈률이 가장 높은 유입 경로는 무엇인가요?  
**Q2.** 유입 경로별 이탈률은 마케팅팀의 행동으로 어떻게 연결할 수 있나요?  
**Q3.** 유입 경로별 이탈률은 좋은 지표의 세 조건을 충족하나요?  
**Q4.** KPI·보조 지표·허무 지표는 어떻게 다른가요? 고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시하세요. 허무 지표는 어떤 맥락에서 성과 판단에 도움이 되지 않는지도 설명하세요.

#### 제출 결과

- `source_metrics` 집계 코드와 결과
- 우선 점검할 유입 경로
- 개선 방향
- 좋은 지표의 조건 평가
- Q1~Q4 답변

In [6]:
# 1~2. referral_source별 그룹화 + 집계
source_metrics = accounts.groupby("referral_source").agg(
    total_accounts=("account_id", "count"),
    churned_accounts=("churn_flag", "sum"),
)
source_metrics["churn_rate"] = source_metrics["churned_accounts"] / source_metrics["total_accounts"]

# 3. 이탈률 높은 순 정렬
source_metrics = source_metrics.sort_values("churn_rate", ascending=False)

# 4. 보기 좋은 형식으로 출력
display_df = source_metrics.copy()
display_df["churn_rate"] = (display_df["churn_rate"] * 100).round(2).astype(str) + "%"
print(display_df)

# 5. 이탈률이 가장 높은 유입 경로
top_churn_source = source_metrics.index[0]
print(f"\n고객사 이탈률이 가장 높은 유입 경로: {top_churn_source} ({source_metrics.iloc[0]['churn_rate']*100:.2f}%)")

                 total_accounts  churned_accounts churn_rate
referral_source                                             
event                        96                29     30.21%
other                       103                25     24.27%
ads                          98                23     23.47%
organic                     114                20     17.54%
partner                      89                13     14.61%

고객사 이탈률이 가장 높은 유입 경로: event (30.21%)


### 과제 1 답변 작성란

**Q1.** 고객사 이탈률이 가장 높은 유입 경로는 무엇인가요?  
-> `"event"의 이탈률이 30.21%로 가장 높음.` 

**Q2.** 유입 경로별 이탈률은 마케팅팀의 행동으로 어떻게 연결할 수 있나요?  
-> `마케팅팀은 event에 투입하는 예산을 조정하거나, event에서 모집한 고객에 한해 온보딩·초기 관리를 강화하는 등`  
`막연한 액션이 아닌 유입 경로별로 구체적인 액션을 취할 수 있음.`  

**Q3.** 유입 경로별 이탈률은 좋은 지표의 세 조건을 충족하나요?
   - `행동 가능성` : 네 / 유입 경로별로 이탈률이 다르게 나오니, 이탈이 높은 특정 경로에 구체적인 조치로 바로 연결됨.
   - `비교 가능성` : 네 / 각 경로별로 "전체 고객사 수 / 이탈 고객사 수 / 이탈률"이 동일하게 나오기 때문에, 서로 비교 가능.
   - `이해 용이성` : 네 / churn_rate만 확인해도 별도 설명 없이 "이 경로가 더 이탈률이 높다"고 이해하기 용이함.

**Q4.** KPI·보조 지표·허무 지표는 어떻게 다른가요?  
고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시하세요.  
허무 지표는 어떤 맥락에서 성과 판단에 도움이 되지 않는지도 설명하세요.  
- `KPI : churn_flag` / '고객사 이탈 여부'를 나타내는 churn_flag는 '이탈 감소'라는 핵심 목표를 가장 직접적으로 측정
- `보조 지표 : referral_source` / '고객사가 유입된 경로'를 확인하여 전체 이탈률(KPI)이 왜 그런 값이 나왔는지,  
어느 유입 경로에서 이탈이 몰리는지 도움을 줄 수 있기 때문에 보조 지표로 선정
- `허무 지표 : subscription_id 개수 (누적 구독 수)` / 이미 끝난 구독과 이탈한 구독들도 포함될 수 있으므로  
누적 구독 수만 보고 서비스가 잘 되고 있다고 판단하면, 실제로는 이탈이 심각한 상황에서도 성과가 좋아 보일 수 있음  
-> '이탈 감소'라는 목표를 판단하는 데는 도움이 되지 않음



---

## 실습 마무리

아래 질문에 답하세요.

1. 이번 실습에서 해결하려고 한 비즈니스 문제는 무엇인가요?  
-> 유로 구독을 안정적으로 유지하면서 반복 매출을 늘리기 위한 핵심 지표를 관리하고  
-> 어느 대상을 우선적으로 개선해야하는지 확인  

2. 현재 목표를 직접 보여주는 KPI로 어떤 지표를 선택했나요?  
-> 현재 활성화된 유료 구독자들이 만드는 매출을 MRR을 선택  

3. KPI의 변화를 이해하기 위해 어떤 보조 지표를 확인했나요?  
-> 전체 구독 이탈률과 요금제별 이탈률을 확인  
-> 이탈률은 현재 또는 미래의 매출 감소로 이어지기 떄문  

4. 허무 지표 후보를 핵심 성과로 사용할 때 어떤 문제가 생길 수 있나요?  
-> 누적 가입 고객사를 핵심 성과로 사용한다면 현재 구독을 하지 않거나,  
-> 활동하지 않아도 실제 성과를 과대평가할 수 있음  

5. 어떤 분석 결과를 근거로 개선 대상을 정했나요?  
-> 요금제별로 MRR과 이탈률을 분석하여 enterprise를 우선 점검 대상으로 정함